### Pipeline 2 — XGBoost

We will now train and evaluate **XGBoost classifiers** using:
- The same feature-engineered datasets
- Cross-validation and hyperparameter tuning
- Undersampling and SMOTE for class balance
- Full evaluation with metrics and confusion matrix

In [11]:
import os
import pandas as pd

# --- Path Config ---
BASE_DIR = os.path.abspath(os.path.join(os.getcwd(), ".."))
DATA_DIR = os.path.join(BASE_DIR, "data")
OUTPUTS_DIR = os.path.join(BASE_DIR, "outputs")
STATS_DIR = os.path.join(OUTPUTS_DIR, "statistics")
PLOTS_DIR = os.path.join(OUTPUTS_DIR, "plots")
METRICS_DIR = os.path.join(OUTPUTS_DIR, "metrics")

# Make sure subfolders exist
os.makedirs(STATS_DIR, exist_ok=True)
os.makedirs(PLOTS_DIR, exist_ok=True)
os.makedirs(METRICS_DIR, exist_ok=True)

part11_plots_dir = os.path.join(PLOTS_DIR, "part11_xgboost_models")
os.makedirs(part11_plots_dir, exist_ok=True)

part11_metrics_dir = os.path.join(METRICS_DIR, "part11_xgboost_models")
os.makedirs(part11_metrics_dir, exist_ok=True)


In [13]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix

from xgboost import XGBClassifier
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.under_sampling import RandomUnderSampler
from imblearn.over_sampling import SMOTE

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

def run_pipeline2(filepath, target_col, handle_unknown=True, output_dir=part11_metrics_dir):
    print(f"\n Running Pipeline 2 (XGBoost) for: {target_col}")
    
    # Load data
    df = pd.read_csv(filepath)
    if handle_unknown:
        df = df[df[target_col] != "Unknown"]

    # Binary encode target
    y = df[target_col].apply(lambda x: 1 if x == "Yes" else 0)
    X = df.drop(columns=[target_col])

    # Train-test split
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, stratify=y, random_state=42
    )

    # Categorical columns
    cat_cols = X_train.select_dtypes(include="object").columns.tolist()

    # Preprocessing
    preprocessor = ColumnTransformer([
        ("cat", OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat_cols)
    ], remainder="passthrough")

    # XGBoost model and parameter grid
    model = XGBClassifier(use_label_encoder=False, eval_metric="logloss")
    param_grid = {
        "model__max_depth": [3, 5, 10],
        "model__learning_rate": [0.01, 0.1, 0.2],
        "model__n_estimators": [50, 100, 200]
    }

    # Clean folder name
    target_folder = target_col.replace(" ", "").replace("(", "").replace(")", "").replace("/", "_")
    full_output_dir = os.path.join(output_dir, target_folder)
    os.makedirs(full_output_dir, exist_ok=True)

    # Results
    metric_rows = []
    matrix_rows = []

    # XGBoost with different sampling strategies
    for sampler_name, sampler in {
        "Undersampling": RandomUnderSampler(random_state=42),
        "SMOTE Oversampling": SMOTE(random_state=42)
    }.items():
        pipeline = ImbPipeline(steps=[
            ("preprocessor", preprocessor),
            ("sampler", sampler),
            ("model", model)
        ])

        # GridSearchCV for hyperparameter tuning
        grid = GridSearchCV(pipeline, param_grid, cv=5, scoring="f1_macro", n_jobs=-1)
        grid.fit(X_train, y_train)

        best_model = grid.best_estimator_
        y_pred = best_model.predict(X_test)
        y_proba = best_model.predict_proba(X_test)[:, 1]

        report = classification_report(y_test, y_pred, output_dict=True, zero_division=0)
        roc_auc = roc_auc_score(y_test, y_proba)
        
        cm = confusion_matrix(y_test, y_pred, labels=[0, 1])
        tn, fp, fn, tp = cm.ravel()

        # cm = confusion_matrix(y_test, y_pred)
        # tn, fp, fn, tp = cm.ravel()

        # Collect metrics
        metric_rows.append({
            "Model": "XGBoost",
            "Sampler": sampler_name,
            "Best Params": grid.best_params_,
            "Accuracy": report["accuracy"],
            "ROC AUC": roc_auc,
            "Precision_Yes (1)": report["1"]["precision"],
            "Recall_Yes (1)": report["1"]["recall"],
            "F1_Yes (1)": report["1"]["f1-score"],
            "Precision_No (0)": report["0"]["precision"],
            "Recall_No (0)": report["0"]["recall"],
            "F1_No (0)": report["0"]["f1-score"],
            "F1_Macro": report["macro avg"]["f1-score"]
        })

        matrix_rows.append({
            "Model": "XGBoost",
            "Sampler": sampler_name,
            "TP": tp, "FP": fp, "TN": tn, "FN": fn
        })

        # Plot confusion matrix
        plt.figure(figsize=(4.5, 4))
        sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False,
                    xticklabels=["No", "Yes"], yticklabels=["No", "Yes"])
        plt.title(f"XGBoost ({sampler_name}) - Confusion Matrix")
        plt.xlabel("Predicted Label")
        plt.ylabel("True Label")
        plt.tight_layout()
        fname = f"{full_output_dir}/conf_matrix_XGBoost_{sampler_name.replace(' ', '_')}.png"
        plt.savefig(fname, bbox_inches='tight')
        plt.close()

        print(f"✅ XGBoost ({sampler_name}) done. F1_Yes = {report['1']['f1-score']:.4f}, ROC AUC = {roc_auc:.4f}")

    # Save results
    metrics_df = pd.DataFrame(metric_rows)
    matrix_df = pd.DataFrame(matrix_rows)

    #plt.savefig(fname, bbox_inches='tight')

    metrics_df.to_csv(os.path.join(full_output_dir, "evaluation_results_XGBoost.csv"), index=False)
    matrix_df.to_csv(os.path.join(full_output_dir, "confusion_matrices_XGBoost.csv"), index=False)


    return metrics_df


In [14]:
df1_results_xgb = run_pipeline2(os.path.join(STATS_DIR, "model1_high_bp.csv"), 
                                target_col="Has a high blood pressure", handle_unknown=False)


 Running Pipeline 2 (XGBoost) for: Has a high blood pressure


d:\Winter\DataScienceWorkplace\Basics\.venv\Lib\site-packages\xgboost\core.py:158: UserWarning: [08:05:52] WARNING: C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-08cbc0333d8d4aae1-1\xgboost\xgboost-ci-windows\src\learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)


✅ XGBoost (Undersampling) done. F1_Yes = 0.6137, ROC AUC = 0.8103


d:\Winter\DataScienceWorkplace\Basics\.venv\Lib\site-packages\xgboost\core.py:158: UserWarning: [08:11:22] WARNING: C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-08cbc0333d8d4aae1-1\xgboost\xgboost-ci-windows\src\learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)


✅ XGBoost (SMOTE Oversampling) done. F1_Yes = 0.6008, ROC AUC = 0.8065


In [15]:
df2_results_xgb = run_pipeline2(os.path.join(STATS_DIR, "model2_diabetes.csv"),
                                target_col="Has diabetes", handle_unknown=True)


 Running Pipeline 2 (XGBoost) for: Has diabetes


d:\Winter\DataScienceWorkplace\Basics\.venv\Lib\site-packages\xgboost\core.py:158: UserWarning: [08:16:06] WARNING: C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-08cbc0333d8d4aae1-1\xgboost\xgboost-ci-windows\src\learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)


✅ XGBoost (Undersampling) done. F1_Yes = 0.3847, ROC AUC = 0.8534


d:\Winter\DataScienceWorkplace\Basics\.venv\Lib\site-packages\xgboost\core.py:158: UserWarning: [08:21:03] WARNING: C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-08cbc0333d8d4aae1-1\xgboost\xgboost-ci-windows\src\learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)


✅ XGBoost (SMOTE Oversampling) done. F1_Yes = 0.4184, ROC AUC = 0.8484


In [16]:
df3_results_xgb = run_pipeline2(os.path.join(STATS_DIR, "model3_cardio.csv"), 
                                target_col="Cardiovascular condition (Heart disease or stroke)", handle_unknown=True)


 Running Pipeline 2 (XGBoost) for: Cardiovascular condition (Heart disease or stroke)


d:\Winter\DataScienceWorkplace\Basics\.venv\Lib\site-packages\xgboost\core.py:158: UserWarning: [08:23:01] WARNING: C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-08cbc0333d8d4aae1-1\xgboost\xgboost-ci-windows\src\learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)


✅ XGBoost (Undersampling) done. F1_Yes = 0.3933, ROC AUC = 0.8559


d:\Winter\DataScienceWorkplace\Basics\.venv\Lib\site-packages\xgboost\core.py:158: UserWarning: [08:28:02] WARNING: C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-08cbc0333d8d4aae1-1\xgboost\xgboost-ci-windows\src\learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)


✅ XGBoost (SMOTE Oversampling) done. F1_Yes = 0.4273, ROC AUC = 0.8493


###  XGBoost Evaluation Summary

While XGBoost provided **comparable performance** to our baseline models, it did **not yield significant improvements** in F1-score or ROC AUC across any target.

Given that **Logistic Regression**:
- Matches or slightly outperforms XGBoost,
- Is **faster**, **simpler**, and **more interpretable**,

We will **retain Logistic Regression with SMOTE Oversampling** as our final model for all three targets.

**No model change needed.**
